# EDS conversion notebook (v2)
This notebook demonstrates the conversion of EDS data from FEMTUS into HyperSpy using the `eds_converter` module. 

### Requirements
- `hyperspy`
- `exspy`
- `h5py`
- `numpy`
- `zarr`

The conversion logic is now decoupled into `Scripts/eds_converter.py` for improved robustness and maintainability.

In [17]:
%matplotlib qt
import sys
import logging
import numpy as np
from pathlib import Path
from zarr import ZipStore
import hyperspy.api as hs

# Add the project root to sys.path so that the Scripts module can be found
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from Scripts.eds_converter import load_EDSjh5

# Configure logging for the converter module
# Change to logging.DEBUG for detailed trace of the conversion process
logging.getLogger('eds_converter').setLevel(logging.INFO)

## Loading
Load the EDS area cube and verify metadata and calibrations

In [18]:
path = Path(r"C:\Users\emilc\OneDrive - NTNU\NORTEM\Data\GrandARM\EDS_Magnus\20260513_Magnus\20260513_Magnus\0513_0907\EDS Area Cube_0513_090754.jh5")
signal = load_EDSjh5(path)
signal

<EDSTEMSpectrum, title: EDS Area Cube_0513_090754, dimensions: (291, 255|4096)>

Plot the data

In [19]:
signal.plot()

Print the axes manager to see the calibrations

In [20]:
signal.axes_manager

Navigation axis name,size,index,offset,scale,units
,291,0,0.0,0.01953125,µm
,255,0,0.0,0.01953125,µm
Signal axis name,size,,offset,scale,units
,4096,,0.0,0.001,MeV


Print the signal metadata

In [7]:
signal.metadata

├── Acquisition_instrument
│   └── TEM
│       ├── Detector
│       │   └── EDS
│       │       ├── azimuth_angle = 0.0
│       │       ├── elevation_angle = 35.0
│       │       ├── energy_resolution_MnKa = 130.0
│       │       ├── live_time = 67.45
│       │       └── real_time = 74.64
│       ├── Stage
│       │   └── tilt_alpha = 0.0
│       └── beam_energy  = 200.0
├── General
│   └── title = EDS Area Cube_0513_090754
└── Signal
    └── signal_type = EDS_TEM

Print the original metadata (the raw metadata stored from FEMTUS)

In [8]:
signal.original_metadata

└── JH5
    ├── Part-0
    │   ├── JifFormatId = 100
    │   ├── JifVersion = 1
    │   ├── Metadata = 
    │   └── OptionalData = 
    ├── Part-1
    │   ├── JifFormatId = 100
    │   ├── JifVersion = 1
    │   ├── Metadata = 
    │   └── OptionalData = 
    └── Summed = <Signal2D, title: EDS Area Cube_0513_090754, dimensions: (|255, 291)>

Print the original metadata of the sum spectrum .jh5 file. This should actually correspond to the metadata for the EDS area cube

In [10]:
signal.original_metadata.JH5.Summed.original_metadata

└── JH5
    ├── JifFormatId = 1069064
    ├── JifVersion = 2
    ├── Metadata
    │   ├── Accelerating Voltage = 
    │   ├── AcqusitionTime = 05/13/2026 09:07:54
    │   ├── CameraLength = 
    │   ├── DataBits = 
    │   ├── DataDetectorTypeID = 2
    │   ├── DataID = 17cbde2b-1d5e-40bd-80fc-50960d4d98eb
    │   ├── DataType = EdsCube
    │   ├── DimensionUnits = 
    │   ├── Magnification = 
    │   ├── MapEndEnergy = 
    │   ├── MapStartEnergy = 
    │   ├── MaxData = 1.79769313486232E+308
    │   ├── MinData = -1.79769313486232E+308
    │   ├── PixelsPerUnit = 
    │   ├── RockingAngle = 
    │   ├── StartEnergy = 
    │   └── Version = 1
    └── OptionalData
        ├── Information
        │   ├── DataInformation
        │   │   ├── Channel = 1
        │   │   ├── ChannelType = GrayScale
        │   │   ├── DataBytes = 4
        │   │   ├── DimensionLength = 2
        │   │   ├── Dimensions = [255, 291]
        │   │   └── TypeInfo = 2
        │   ├── Header
        │   │   ├── ClumpId = 8a05c577-193a-4312-93e6-e4f8ebd51835
        │   │   ├── ClumpType = EdsCube
        │   │   ├── DataSubType = 
        │   │   ├── DataType = EDS_Cube
        │   │   ├── DetectorType = 0
        │   │   ├── Name = EDS Area Cube_0513_090754
        │   │   └── Version = 1.0.0
        │   ├── MeasurementInformation
        │   │   └── CalibrationCoefficients <list>
        │   │       ╠══ [0]
        │   │       ║   ╠══ Offset = 0.0
        │   │       ║   ╠══ Scale = 19.53125
        │   │       ║   ╚══ Unit = Nanometer
        │   │       ╠══ [1]
        │   │       ║   ╠══ Offset = 0.0
        │   │       ║   ╠══ Scale = 19.53125
        │   │       ║   ╚══ Unit = Nanometer
        │   │       ╚══ [2]
        │   │           ╠══ Offset = 0.0
        │   │           ╠══ Scale = 1
        │   │           ╚══ Unit = KeV
        │   └── Tags
        │       ├── Aperture
        │       │   └── ApertureHoleString = 1
        │       ├── DataAdditionalInformation
        │       │   ├── FFT = False
        │       │   └── ReciprocalSpace = False
        │       ├── Detector
        │       │   ├── BinningSize
        │       │   │   ├── X = 1
        │       │   │   └── Y = 1
        │       │   ├── DetectorKind = DFI_U
        │       │   ├── ExposureTimeIndex = 
        │       │   ├── ExposureTimeValue = 1.0
        │       │   ├── FrameIntegration = 1
        │       │   ├── GainIndex = 
        │       │   ├── ImagingArea
        │       │   │   ├── Height = 256
        │       │   │   ├── Width = 2048
        │       │   │   ├── X = 0
        │       │   │   └── Y = 0
        │       │   ├── Manufacturer = JEOL Ltd.
        │       │   ├── ModelCode = 
        │       │   ├── ModelDisplayName = 
        │       │   ├── OffsetIndex = 
        │       │   └── PixelsPerMeter
        │       │       ├── Horizontal = 2560.0
        │       │       └── Vertical = 2560.0
        │       ├── EDS
        │       │   ├── DetectorInformation
        │       │   │   └── FirstDetector
        │       │   │       ├── DetectorName = First
        │       │   │       ├── DetectorType = EX-34400MNU
        │       │   │       ├── InputCountrate = 9533
        │       │   │       ├── LiveTime = 67.45
        │       │   │       ├── OutputCountrate = 7868
        │       │   │       ├── PortType = ED2
        │       │   │       ├── ProcessTimeTemplateName = T4
        │       │   │       └── RealTime = 74.64
        │       │   └── SpectrumEnergyChannel = 2048
        │       ├── EOS
        │       │   ├── CalibratedCameraLength = 150.0
        │       │   ├── CalibratedMagnificationValue = 20000.0
        │       │   ├── CameraLength = 150.0
        │       │   ├── CameraLengthCalibrated = False
        │       │   ├── CameraLengthString = 15cm
        │       │   ├── ConvergenceAngleAlphaNumber = 2
        │       │   ├── ConvergenceAngleAlphaString = 2
        │       │   ├── ConvergenceAngleAlphaUnitString = 2
        │       │   ├── ConvergenceA

## Save the signal to zspy
Note: Don't mind the many UserWarnings that will appear

In [11]:
store = ZipStore(path.with_suffix('.zspy'), mode='w')
signal.save(store)

c:\Users\emilc\AppData\Local\miniforge3\envs\pyxem0.21.0\Lib\zipfile\__init__.py:1642: UserWarning: Duplicate name: '.zattrs'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
c:\Users\emilc\AppData\Local\miniforge3\envs\pyxem0.21.0\Lib\zipfile\__init__.py:1642: UserWarning: Duplicate name: 'Experiments/EDS Area Cube_0513_090754/axis-0/.zattrs'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
c:\Users\emilc\AppData\Local\miniforge3\envs\pyxem0.21.0\Lib\zipfile\__init__.py:1642: UserWarning: Duplicate name: 'Experiments/EDS Area Cube_0513_090754/axis-1/.zattrs'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
c:\Users\emilc\AppData\Local\miniforge3\envs\pyxem0.21.0\Lib\zipfile\__init__.py:1642: UserWarning: Duplicate name: 'Experiments/EDS Area Cube_0513_090754/axis-2/.zattrs'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
c:\Users\emilc\AppData\Local\miniforge3\envs\pyxem0.21.0\Lib\zipfile\__init__.py:1642: UserWarning: Duplicate name: '

## Load the saved zspy signal again to verify

In [12]:
store = ZipStore(path.with_suffix('.zspy'), mode='r')
s_verify = hs.load(store, lazy=True)
s_verify.metadata

├── Acquisition_instrument
│   └── TEM
│       ├── Detector
│       │   └── EDS
│       │       ├── azimuth_angle = 0.0
│       │       ├── elevation_angle = 35.0
│       │       ├── energy_resolution_MnKa = 130.0
│       │       ├── live_time = 67.45
│       │       └── real_time = 74.64
│       ├── Stage
│       │   └── tilt_alpha = 0.0
│       └── beam_energy  = 200.0
├── General
│   ├── FileIO
│   │   ├── 0
│   │   │   ├── hyperspy_version = 2.3.0
│   │   │   ├── io_plugin = rsciio.zspy
│   │   │   ├── operation = save
│   │   │   └── timestamp = 2026-06-25T12:52:59.684222+02:00
│   │   └── 1
│   │       ├── hyperspy_version = 2.3.0
│   │       ├── io_plugin = rsciio.zspy
│   │       ├── operation = load
│   │       └── timestamp = 2026-06-25T12:53:03.126147+02:00
│   └── title = EDS Area Cube_0513_090754
└── Signal
    └── signal_type = EDS_TEM

In [13]:
s_verify.axes_manager

Navigation axis name,size,index,offset,scale,units
,291,0,0.0,0.01953125,µm
,255,0,0.0,0.01953125,µm
Signal axis name,size,,offset,scale,units
,4096,,0.0,0.001,MeV


In [14]:
s_verify.plot()

  0%|          | 0/33 [00:00<?, ?it/s]

## Interactive operations
### Subarea analysis of the EDS area cube

In [15]:
analysis = 'mean' #Choose between 'sum', 'mean', 'std', 'max', 'min'
s_verify.plot()
roi = hs.roi.RectangularROI()
signal_roi = roi.interactive(s_verify, color='red')
analysis_options = {'sum': signal_roi.sum, 'mean': signal_roi.mean, 'std': signal_roi.std, 'max': signal_roi.max, 'min': signal_roi.min}
roi_spectrum = hs.interactive(analysis_options[analysis], event=roi.events.changed, recompute_out_event=None)
roi_spectrum.plot()

  0%|          | 0/23 [00:00<?, ?it/s]

### Energy windows

In [16]:
hs.plot.plot_roi_map(s_verify, rois=2, navigator=s_verify.sum())

([SpanROI(left=0, right=1.02375), SpanROI(left=1.02375, right=2.0475)],
 [<LazySignal2D, title: Integrated intensity, dimensions: (|291, 255)>,
  <LazySignal2D, title: Integrated intensity, dimensions: (|291, 255)>])